# B2 · Lasso and noise features: automatic selection and scale

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Add 30 pure noise features to the diabetes dataset, use `LassoCV` to see which ones survive, and find out
how the result changes if you forget the scaler.

## Dataset

Diabetes (`ds.diabetes()`): 442 patients, 10 features **already standardized by scikit-learn** (standard deviation
about 0.05) and as target the progression of the disease. In the setup cell we put next to them 30 columns
`noise_i` of random numbers with standard deviation 1, which by construction have no relationship with `y`.

## Educational purpose

Use `Pipeline`, `StandardScaler` and `LassoCV` the right way and get a hands-on feel for the most common mistake with
regularization: the penalty depends on the scale of the features. Here the scale is deliberately unbalanced in
favor of the noise.

## Task

1. Split into training and test (75/25, `random_state=0`). Compute `rmse_ols_real` (OLS on the 10 true features only) and `rmse_ols_all` (OLS on all 40).
2. Build `lasso_scaled = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0, max_iter=20000))`, train it and save `rmse_lasso`. Count how many **true** features (`n_real_kept`) and how many **noise** ones (`n_noise_kept`) have a nonzero coefficient.
3. Repeat without the scaler: `lasso_raw = LassoCV(cv=5, random_state=0, max_iter=20000)`. Save `n_real_kept_raw` and `n_noise_kept_raw`.
4. Look at `X_tr.std()` and choose in `answer` ("a", "b" or "c") the correct explanation of the result of step 3:
   - **a**: without the scaler the Lasso penalizes more the features with a large standard deviation, so it discards the noise;
   - **b**: the features with a small standard deviation need large coefficients to have the same effect, so the penalty hits them harder; the noise, with standard deviation 1, has small coefficients and escapes;
   - **c**: `LassoCV` without the scaler does not converge because it requires zero-mean data.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_real, y = ds.diabetes()
rng = np.random.default_rng(0)
noise = pd.DataFrame(rng.normal(size=(len(X_real), 30)), columns=[f"noise_{i}" for i in range(30)])
X = pd.concat([X_real, noise], axis=1)          # 10 true features + 30 of pure noise
real_columns = list(X_real.columns)
X.shape

## Your work

In [ ]:
X_tr, X_te, y_tr, y_te = ...

def rmse(model, columns):
    """Test-set RMSE of an already trained model, using only the given `columns`."""
    ...

rmse_ols_real = ...
rmse_ols_all = ...

In [ ]:
lasso_scaled = ...
rmse_lasso = ...
coef = lasso_scaled[-1].coef_
n_real_kept = ...
n_noise_kept = ...

In [ ]:
lasso_raw = ...
n_real_kept_raw = ...
n_noise_kept_raw = ...

In [ ]:
print(X_tr.std().round(3).to_string())
answer = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='To count the nonzero coefficients of the true features: `(coef[:10] != 0).sum()`; for the noise: `(coef[10:] != 0).sum()` (the column order is 10 true, then 30 of noise). A model inside a `Pipeline` is indexed with `[-1]` to take the last step.'):
    check(rmse_ols_all > rmse_ols_real, "adding noise makes OLS worse",
          "OLS on 40 features should have a higher RMSE than the one on 10")
    check(rmse_lasso < rmse_ols_all, "the scaled Lasso beats OLS on the 40 features",
          "with scaler and Lasso you should do better than OLS on all the features")
    check(n_real_kept >= 3, "the Lasso keeps several true features",
          "the scaled Lasso should keep at least a few true features")
    check(n_noise_kept < 10, "the scaled Lasso discards most of the noise",
          "with the scaler the Lasso should discard almost all of the 30 noise features")
    check(n_noise_kept_raw >= 25, "without the scaler the Lasso keeps almost all the noise",
          "without the scaler almost all of the 30 noise features should survive: did you remove the StandardScaler?")
    check(n_real_kept_raw + n_noise_kept_raw > 0, "the raw model has been trained",
          "count the nonzero coefficients of lasso_raw.coef_")
    check(answer == "b", "correct explanation", "reread the standard deviation of the true features and of the noise")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_lasso_and_noise_features.ipynb`. Open it after you pass the check, or after you have really tried.